# Cluster Existing Proposal Verification Demo

This notebook verifies already-generated DINO+SAM2 proposals from one GUI cluster.
It does **not** rerun DINO/SAM2, does **not** recluster, and does **not** overwrite existing GUI outputs.

It adds objectness checking, crop-level OWLv2 verification, CLIP/SigLIP suggestion, label fusion, qwen-needed export, debug visualizations, CSV, JSONL, and summary tables.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations
import ast, json, math, random, traceback
from pathlib import Path
from typing import Any
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

try:
    import cv2
except Exception:
    cv2 = None

try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    torch = None
    device = "cpu"

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    return start

REPO_ROOT = find_repo_root()

metadata_path = "data/auto_label_demo/embeddings/object_metadata.parquet"
cluster_summary_path = "data/auto_label_demo/clustering/cluster_summary.csv"
output_root = "outputs/cluster_existing_proposal_verification_demo"

selected_cluster_key = "PUT_CLUSTER_KEY_HERE"
selected_cluster_id = None
max_samples = 50
RANDOM_SEED = 42

RUN_OWLV2 = True
RUN_CLIP_OR_SIGLIP = True
RUN_QWEN = False
USE_MASKED_CROP = True
SAVE_DEBUG_VIS = True
LOCAL_FILES_ONLY = True

OWLV2_MODEL_ID = str(REPO_ROOT / "models" / "owlv2-base-patch16") if (REPO_ROOT / "models" / "owlv2-base-patch16").exists() else "google/owlv2-base-patch16"
VERIFIER_MODEL_ID = "openai/clip-vit-base-patch32"

OUTPUT_ROOT = Path(output_root)
DEBUG_DIR = OUTPUT_ROOT / "debug_vis"
QWEN_DIR = OUTPUT_ROOT / "qwen_needed"
for p in [OUTPUT_ROOT, DEBUG_DIR, QWEN_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("device:", device)
print("repo root:", REPO_ROOT)
print("Edit before running if needed: metadata_path, cluster_summary_path, selected_cluster_key/selected_cluster_id, output_root")

## 2. Load metadata and select one cluster

In [ ]:
def repo_path(path: str | Path) -> Path:
    path = Path(path)
    if path.is_absolute():
        return path
    if path.exists():
        return path
    return REPO_ROOT / path

def read_table(path: str | Path) -> pd.DataFrame:
    path = repo_path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".json"}:
        with path.open("r", encoding="utf-8") as fh:
            return pd.DataFrame(json.loads(line) for line in fh if line.strip())
    raise ValueError(f"Unsupported table format: {path}")

metadata_path_resolved = repo_path(metadata_path)
if not metadata_path_resolved.exists():
    for alt in [
        repo_path("data/auto_label_demo/embeddings/object_metadata_clustered.parquet"),
        repo_path("data/auto_label_demo/embeddings/object_metadata_clustered.csv"),
        repo_path("data/auto_label_demo/embeddings/object_metadata.csv"),
    ]:
        if alt.exists():
            print("metadata fallback:", alt)
            metadata_path_resolved = alt
            break

cluster_summary_path_resolved = repo_path(cluster_summary_path)
if not cluster_summary_path_resolved.exists():
    for alt in [
        repo_path("data/auto_label_demo/cluster_review/cluster_summary.csv"),
        repo_path("data/auto_label_demo/cluster_review/clusters.csv"),
    ]:
        if alt.exists():
            print("cluster summary fallback:", alt)
            cluster_summary_path_resolved = alt
            break

metadata = read_table(metadata_path_resolved)
cluster_summary = read_table(cluster_summary_path_resolved) if cluster_summary_path_resolved.exists() else pd.DataFrame()

print("metadata shape:", metadata.shape)
print("available metadata columns:", list(metadata.columns))
print("cluster summary shape:", cluster_summary.shape)
print("available cluster summary columns:", list(cluster_summary.columns))

if not cluster_summary.empty and "cluster_id" in metadata.columns and "cluster_id" in cluster_summary.columns:
    cols = [c for c in ["cluster_id", "cluster_key", "suggested_label", "cluster_suggested_label", "cluster_label", "display_label", "human_label"] if c in cluster_summary.columns]
    metadata = metadata.merge(cluster_summary[cols].drop_duplicates("cluster_id"), on="cluster_id", how="left", suffixes=("", "_cluster_summary"))

if selected_cluster_key != "PUT_CLUSTER_KEY_HERE" and "cluster_key" in metadata.columns:
    cluster_df = metadata[metadata["cluster_key"].astype(str) == str(selected_cluster_key)].copy()
elif selected_cluster_id is not None and "cluster_id" in metadata.columns:
    cluster_df = metadata[pd.to_numeric(metadata["cluster_id"], errors="coerce") == int(selected_cluster_id)].copy()
else:
    if "cluster_id" not in metadata.columns:
        raise ValueError("Set selected_cluster_key or selected_cluster_id; metadata has no cluster_id.")
    selected_cluster_id = int(metadata["cluster_id"].value_counts().index[0])
    cluster_df = metadata[pd.to_numeric(metadata["cluster_id"], errors="coerce") == selected_cluster_id].copy()
    print("No cluster selected; using largest cluster_id for demo:", selected_cluster_id)

if cluster_df.empty:
    raise ValueError("Selected cluster has no rows. Edit selected_cluster_key or selected_cluster_id.")

print("selected cluster size:", len(cluster_df))
for col in ["label", "predicted_label", "raw_label", "dino_label", "detector_label"]:
    if col in cluster_df.columns:
        print("detector label distribution", col, cluster_df[col].fillna("").astype(str).value_counts().head(10).to_dict())
for col in ["suggested_label", "cluster_suggested_label", "cluster_label", "display_label", "human_label"]:
    if col in cluster_df.columns:
        print("cluster/human label distribution", col, cluster_df[col].fillna("").astype(str).value_counts().head(10).to_dict())
for col in ["confidence", "detector_confidence", "dino_score", "score"]:
    if col in cluster_df.columns:
        print("confidence stats", col)
        display(pd.to_numeric(cluster_df[col], errors="coerce").describe())

## 3. Column normalization

In [ ]:
def first_existing(row, names, default=None):
    for name in names:
        if name in row and pd.notna(row[name]) and row[name] != "":
            return row[name]
    return default

def parse_maybe_list(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, (list, tuple, np.ndarray)):
        return [float(x) for x in value[:4]]
    text = str(value).strip()
    if not text:
        return None
    for parser in (json.loads, ast.literal_eval):
        try:
            out = parser(text)
            if isinstance(out, (list, tuple)) and len(out) >= 4:
                return [float(x) for x in out[:4]]
        except Exception:
            pass
    return None

def resolve_path(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    text = str(value).strip()
    if not text:
        return None
    p = Path(text)
    if p.exists():
        return str(p)
    p = REPO_ROOT / text
    return str(p) if p.exists() else text

def normalize_row(row, idx):
    bbox = parse_maybe_list(first_existing(row, ["bbox_xyxy", "bbox"]))
    if bbox is None and all(c in row for c in ["x1", "y1", "x2", "y2"]):
        bbox = [float(row["x1"]), float(row["y1"]), float(row["x2"]), float(row["y2"])]
    return {
        "norm_proposal_id": int(first_existing(row, ["proposal_id", "embedding_idx"], idx)),
        "norm_frame_path": resolve_path(first_existing(row, ["frame_path", "image_path"])),
        "norm_bbox_xyxy": bbox,
        "norm_mask_path": resolve_path(first_existing(row, ["mask_path"])),
        "norm_crop_path": resolve_path(first_existing(row, ["crop_path"])),
        "norm_dino_label": str(first_existing(row, ["dino_label", "predicted_label", "detector_label", "raw_label", "label"], "unknown") or "unknown"),
        "norm_dino_score": float(first_existing(row, ["dino_score", "detector_confidence", "confidence", "score"], 0.0) or 0.0),
        "norm_cluster_suggested_label": str(first_existing(row, ["cluster_suggested_label", "cluster_label", "suggested_label", "display_label"], "") or ""),
        "norm_human_label": str(first_existing(row, ["human_label"], "") or ""),
        "norm_cluster_id": first_existing(row, ["cluster_id"], None),
        "norm_cluster_key": str(first_existing(row, ["cluster_key"], "") or ""),
    }

norm_df = pd.concat(
    [cluster_df.reset_index(drop=True), pd.DataFrame([normalize_row(r, i) for i, (_, r) in enumerate(cluster_df.iterrows())])],
    axis=1,
)
print("missing frame/crop/mask/bbox:", norm_df["norm_frame_path"].isna().sum(), norm_df["norm_crop_path"].isna().sum(), norm_df["norm_mask_path"].isna().sum(), norm_df["norm_bbox_xyxy"].isna().sum())
display(norm_df[["norm_proposal_id", "norm_dino_label", "norm_dino_score", "norm_cluster_suggested_label"]].head())

## 4. Sample diverse examples

In [ ]:
def diverse_sample(df, n):
    if len(df) <= n:
        return df.reset_index(drop=True)
    parts = []
    score = pd.to_numeric(df["norm_dino_score"], errors="coerce").fillna(0)
    parts += [df.loc[score.nlargest(max(1, n // 5)).index], df.loc[score.nsmallest(max(1, n // 5)).index]]
    for _, g in df.groupby("norm_dino_label", dropna=False):
        parts.append(g.sample(min(len(g), 2), random_state=RANDOM_SEED))
    if "quality_status" in df.columns:
        for _, g in df.groupby("quality_status", dropna=False):
            parts.append(g.sample(min(len(g), 2), random_state=RANDOM_SEED))
    out = pd.concat(parts).drop_duplicates("norm_proposal_id")
    if len(out) < n:
        rest = df[~df["norm_proposal_id"].isin(out["norm_proposal_id"])]
        out = pd.concat([out, rest.sample(min(len(rest), n - len(out)), random_state=RANDOM_SEED)])
    return out.head(n).reset_index(drop=True)

sample_df = diverse_sample(norm_df, max_samples)
print("sampled count:", len(sample_df), "of", len(norm_df))

## 5. Label hierarchy

In [ ]:
FINE_TO_COARSE = {}
def add(labels, coarse):
    for label in labels:
        FINE_TO_COARSE[label] = coarse
add(["hand", "glove"], "hand")
add(["pot", "pan", "lid", "cookware", "tray", "kettle"], "cookware")
add(["bowl", "plate", "cup", "glass", "bottle", "jar"], "dishware")
add(["container", "box", "package", "bag", "carton", "can"], "container")
add(["knife", "fork", "spoon", "spatula", "tongs", "ladle", "whisk", "peeler", "scissors", "cutting board"], "utensil")
add(["pasta", "noodles", "rice", "bread", "vegetable", "fruit", "meat", "fish", "egg", "cheese", "ingredient", "food", "dry food", "liquid", "water", "milk", "sauce", "oil", "powder", "sugar", "salt", "olive"], "ingredient")
add(["sink", "faucet", "stove", "cooktop", "oven", "microwave", "fridge", "drawer", "cabinet", "countertop", "table", "rack", "sponge", "towel"], "kitchen_scene")
add(["screwdriver", "wrench", "drill", "pliers", "tool"], "tool")
add(["screw", "bolt", "nut", "washer", "fastener"], "fastener")
add(["panel", "cover", "bracket", "component", "part"], "part")
add(["cable", "wire", "connector", "plug", "socket"], "connector")
add(["bin"], "container")
add(["workbench", "machine surface", "surface"], "surface")
add(["unknown"], "unknown")
NEGATIVE_LABELS = ["background", "shadow", "reflection", "surface", "texture", "wall", "floor", "unclear object", "none of the above"]
POSITIVE_LABELS = sorted([k for k in FINE_TO_COARSE if k != "unknown"])
CANDIDATE_LABELS = POSITIVE_LABELS + NEGATIVE_LABELS

def normalize_label(label):
    text = str(label or "").strip().lower()
    if ":" in text:
        text = text.split(":", 1)[-1].strip()
    return text or "unknown"
def fine_to_coarse(label):
    fine = normalize_label(label)
    if fine in NEGATIVE_LABELS or fine in {"background", "background_candidate"}:
        return "background"
    return FINE_TO_COARSE.get(fine, "unknown")
def display_label(fine):
    fine = normalize_label(fine)
    return f"{fine_to_coarse(fine)}:{fine}"

## 6. Load image, crop, and mask

In [ ]:
def load_image_path(path):
    if not path:
        return None
    p = Path(str(path))
    if not p.exists():
        return None
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None
def load_frame(row): return load_image_path(row.get("norm_frame_path"))
def get_existing_bbox(row):
    bbox = row.get("norm_bbox_xyxy")
    return bbox if isinstance(bbox, list) and len(bbox) >= 4 else None
def make_crop_from_bbox(frame, bbox):
    w, h = frame.size
    x1, y1, x2, y2 = [int(round(float(v))) for v in bbox]
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
    return frame.crop((x1, y1, x2, y2)) if x2 > x1 and y2 > y1 else None
def load_existing_crop(row):
    crop = load_image_path(row.get("norm_crop_path"))
    if crop is not None:
        return crop
    frame, bbox = load_frame(row), get_existing_bbox(row)
    return make_crop_from_bbox(frame, bbox) if frame is not None and bbox is not None else None
def load_existing_mask(row):
    p = row.get("norm_mask_path")
    if not p or not Path(str(p)).exists():
        return None
    try:
        return np.array(Image.open(p).convert("L")) > 0
    except Exception:
        return None
def make_masked_crop(frame, bbox, mask):
    crop = make_crop_from_bbox(frame, bbox)
    if crop is None or mask is None:
        return crop
    x1, y1, x2, y2 = [int(round(float(v))) for v in bbox]
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(frame.size[0], x2), min(frame.size[1], y2)
    m = mask[y1:y2, x1:x2]
    arr = np.array(crop).copy()
    if m.shape[:2] == arr.shape[:2]:
        arr[~m] = 96
    return Image.fromarray(arr)

## 7. Objectness Gate

In [ ]:
def objectness_gate(row):
    frame, crop, mask, bbox = load_frame(row), load_existing_crop(row), load_existing_mask(row), get_existing_bbox(row)
    flags, reasons = [], []
    if bbox is None and crop is None:
        return {"objectness_score": 0.0, "objectness_status": "fake_object", "objectness_reasons": ["bbox and crop missing"], "objectness_flags": ["missing_geometry"], "mask_quality_score": 0.0, "mask_quality_pass": False}
    image_area = float(frame.size[0] * frame.size[1]) if frame is not None else float(crop.size[0] * crop.size[1])
    if bbox is not None:
        bw, bh = max(1.0, bbox[2]-bbox[0]), max(1.0, bbox[3]-bbox[1])
    else:
        bw, bh = crop.size
    bbox_area = bw * bh
    bbox_area_ratio = bbox_area / max(1.0, image_area)
    aspect = max(bw / bh, bh / bw)
    if bbox_area_ratio < 0.00005: flags.append("bbox_extremely_tiny"); reasons.append("bbox area extremely tiny")
    if aspect > 8: flags.append("bbox_aspect_too_large"); reasons.append("bbox aspect ratio > 8")
    mask_area_ratio_image = mask_area_ratio_bbox = components = border_touch_ratio = contrast = None
    if mask is not None:
        mask_area = float(mask.sum())
        mask_area_ratio_image = mask_area / max(1.0, image_area)
        mask_area_ratio_bbox = mask_area / max(1.0, bbox_area)
        if cv2 is not None:
            components = int(max(0, cv2.connectedComponents(mask.astype("uint8"), 8)[0] - 1))
        border = np.zeros(mask.shape, dtype=bool); border[:2,:]=border[-2:,:]=border[:,:2]=border[:,-2:]=True
        border_touch_ratio = float((mask & border).sum()) / max(1.0, mask_area)
        if mask_area_ratio_image < 0.0002: flags.append("mask_area_too_small"); reasons.append("mask area too small")
        if mask_area_ratio_image > 0.65: flags.append("mask_area_too_large"); reasons.append("mask area too large")
        if mask_area_ratio_bbox < 0.15: flags.append("mask_bbox_area_too_sparse"); reasons.append("mask / bbox area too sparse")
        if border_touch_ratio > 0.45: flags.append("severe_border_touch"); reasons.append("severe border touch")
        mask_quality_score = max(0, min(1, 0.7 + (mask_area_ratio_bbox or 0) * 0.2 - 0.15 * len(flags)))
    else:
        reasons.append("mask missing; bbox/crop only")
        mask_quality_score = 0.5
    score = max(0, min(1, 0.8 - 0.18 * len(flags) + (0.05 if mask is not None else -0.05)))
    status = "real_object" if score >= 0.70 else "uncertain_object" if score >= 0.40 else "fake_object"
    return {"bbox_area_ratio_image": bbox_area_ratio, "bbox_aspect_ratio": aspect, "mask_available": mask is not None, "mask_area_ratio_image": mask_area_ratio_image, "mask_area_ratio_bbox": mask_area_ratio_bbox, "mask_connected_components": components, "border_touch_ratio": border_touch_ratio, "foreground_background_contrast": contrast, "objectness_score": score, "objectness_status": status, "objectness_reasons": reasons, "objectness_flags": flags, "mask_quality_score": mask_quality_score, "mask_quality_pass": mask_quality_score >= 0.45 and status != "fake_object"}

## 8. OWLv2 crop-level verifier

In [ ]:
owlv2_state = {"available": False, "error": "not loaded"}
if RUN_OWLV2:
    try:
        from transformers import Owlv2ForObjectDetection, Owlv2Processor
        owlv2_processor = Owlv2Processor.from_pretrained(OWLV2_MODEL_ID, local_files_only=LOCAL_FILES_ONLY)
        owlv2_model = Owlv2ForObjectDetection.from_pretrained(OWLV2_MODEL_ID, local_files_only=LOCAL_FILES_ONLY).to(device).eval()
        owlv2_state = {"available": True, "processor": owlv2_processor, "model": owlv2_model, "error": ""}
        print("OWLv2 loaded:", OWLV2_MODEL_ID)
    except Exception as exc:
        owlv2_state = {"available": False, "error": f"{type(exc).__name__}: {exc}"}
        print("OWLv2 unavailable:", owlv2_state["error"])

def run_owlv2_crop_verifier(crop, candidate_labels):
    out = {"owlv2_available": owlv2_state["available"], "owlv2_crop_top1_label": None, "owlv2_crop_top1_score": None, "owlv2_crop_top5_labels": [], "owlv2_crop_top5_scores": [], "owlv2_crop_coarse_label": None, "owlv2_positive_detected": False, "owlv2_top1_is_negative": None}
    if not owlv2_state["available"] or crop is None:
        return out
    try:
        labels = [normalize_label(x) for x in candidate_labels]
        inputs = owlv2_state["processor"](text=[labels], images=crop.convert("RGB"), return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():
            pred = owlv2_state["model"](**inputs)
        target_sizes = torch.tensor([(crop.height, crop.width)], device=device)
        proc = owlv2_state["processor"].post_process_object_detection(pred, threshold=0.01, target_sizes=target_sizes)[0]
        scores = {}
        for s, li in zip(proc["scores"], proc["labels"]):
            lab = labels[int(li.detach().cpu())]
            scores[lab] = max(scores.get(lab, 0), float(s.detach().cpu()))
        top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]
        if top:
            out.update({"owlv2_crop_top1_label": top[0][0], "owlv2_crop_top1_score": top[0][1], "owlv2_crop_top5_labels": [x[0] for x in top], "owlv2_crop_top5_scores": [x[1] for x in top], "owlv2_crop_coarse_label": fine_to_coarse(top[0][0]), "owlv2_positive_detected": top[0][0] not in NEGATIVE_LABELS, "owlv2_top1_is_negative": top[0][0] in NEGATIVE_LABELS})
    except Exception as exc:
        out["owlv2_error"] = f"{type(exc).__name__}: {exc}"
    return out

## 9. CLIP or SigLIP top-k suggester

In [ ]:
verifier_state = {"available": False, "model_name": VERIFIER_MODEL_ID, "error": "not loaded"}
if RUN_CLIP_OR_SIGLIP:
    try:
        from transformers import AutoModel, AutoProcessor
        verifier_processor = AutoProcessor.from_pretrained(VERIFIER_MODEL_ID, local_files_only=LOCAL_FILES_ONLY)
        verifier_model = AutoModel.from_pretrained(VERIFIER_MODEL_ID, local_files_only=LOCAL_FILES_ONLY).to(device).eval()
        verifier_state = {"available": True, "processor": verifier_processor, "model": verifier_model, "model_name": VERIFIER_MODEL_ID, "error": ""}
        print("Verifier loaded:", VERIFIER_MODEL_ID)
    except Exception as exc:
        verifier_state = {"available": False, "model_name": VERIFIER_MODEL_ID, "error": f"{type(exc).__name__}: {exc}"}
        print("Verifier unavailable:", verifier_state["error"])

PROMPT_TEMPLATES = ["a photo of a {}", "a cropped photo of a {}", "a close-up photo of a {}"]

def verify_crop_zero_shot(crop, candidate_labels):
    out = {"verifier_available": verifier_state["available"], "verifier_model_name": verifier_state["model_name"], "verifier_top1_label": None, "verifier_top1_score": None, "verifier_top5_labels": [], "verifier_top5_scores": [], "verifier_margin": None, "verifier_coarse_label": None, "verifier_top1_is_negative": None, "verifier_positive_best_label": None, "verifier_positive_best_score": None, "verifier_positive_best_coarse_label": None}
    if not verifier_state["available"] or crop is None:
        return out
    try:
        labels = [normalize_label(x) for x in candidate_labels]
        texts = [tmpl.format(label) for label in labels for tmpl in PROMPT_TEMPLATES]
        proc, model = verifier_state["processor"], verifier_state["model"]
        img_inputs = proc(images=crop.convert("RGB"), return_tensors="pt").to(device)
        txt_inputs = proc(text=texts, padding=True, return_tensors="pt").to(device)
        with torch.inference_mode():
            img = model.get_image_features(**img_inputs) if hasattr(model, "get_image_features") else model(**img_inputs).pooler_output
            txt = model.get_text_features(**txt_inputs) if hasattr(model, "get_text_features") else model(**txt_inputs).pooler_output
        img = img / img.norm(dim=-1, keepdim=True); txt = txt / txt.norm(dim=-1, keepdim=True)
        sims = (img @ txt.T)[0].detach().float().cpu().numpy()
        label_scores = [(lab, float(np.mean(sims[i*len(PROMPT_TEMPLATES):(i+1)*len(PROMPT_TEMPLATES)]))) for i, lab in enumerate(labels)]
        top = sorted(label_scores, key=lambda x: x[1], reverse=True)[:5]
        pos = [x for x in label_scores if x[0] not in NEGATIVE_LABELS]
        best_pos = max(pos, key=lambda x: x[1]) if pos else (None, None)
        out.update({"verifier_top1_label": top[0][0], "verifier_top1_score": top[0][1], "verifier_top5_labels": [x[0] for x in top], "verifier_top5_scores": [x[1] for x in top], "verifier_margin": top[0][1] - top[1][1] if len(top) > 1 else 0, "verifier_coarse_label": fine_to_coarse(top[0][0]), "verifier_top1_is_negative": top[0][0] in NEGATIVE_LABELS, "verifier_positive_best_label": best_pos[0], "verifier_positive_best_score": best_pos[1], "verifier_positive_best_coarse_label": fine_to_coarse(best_pos[0]) if best_pos[0] else None})
    except Exception as exc:
        out["verifier_error"] = f"{type(exc).__name__}: {exc}"
    return out

## 10. Cluster evidence and label fusion

In [ ]:

HIGH_RISK_EDGE_LABELS = {"knife", "fork", "spoon", "spatula", "tongs", "ladle", "whisk", "peeler", "scissors", "cutting board"}

# These thresholds are intentionally conservative. Agreement from weak crop verifiers
# should not turn a noisy edge fragment into a high-quality label.
MIN_OWLV2_SUPPORT_SCORE = 0.45
MIN_VERIFIER_SUPPORT_SCORE = 0.32
MIN_VERIFIER_SUPPORT_MARGIN = 0.025
MIN_DINO_HIGH_CONF = 0.40


def _is_noise_cluster(value) -> bool:
    try:
        return int(float(value)) == -1
    except Exception:
        return str(value).strip().lower() in {"-1", "noise", "outlier"}


def _strong_owlv2_support(e, expected_coarse: str) -> bool:
    label = normalize_label(e.get("owlv2_crop_top1_label")) if e.get("owlv2_crop_top1_label") else ""
    score = float(e.get("owlv2_crop_top1_score") or 0.0)
    return bool(label and label not in NEGATIVE_LABELS and fine_to_coarse(label) == expected_coarse and score >= MIN_OWLV2_SUPPORT_SCORE)


def _strong_verifier_support(e, expected_coarse: str) -> bool:
    label = normalize_label(e.get("verifier_positive_best_label") or e.get("verifier_top1_label")) if (e.get("verifier_positive_best_label") or e.get("verifier_top1_label")) else ""
    score = float(e.get("verifier_positive_best_score") or e.get("verifier_top1_score") or 0.0)
    margin = float(e.get("verifier_margin") or 0.0)
    return bool(label and label not in NEGATIVE_LABELS and fine_to_coarse(label) == expected_coarse and score >= MIN_VERIFIER_SUPPORT_SCORE and margin >= MIN_VERIFIER_SUPPORT_MARGIN)


def choose_final_label(e):
    reasons = []
    dino = normalize_label(e.get("dino_label")); dino_score = float(e.get("dino_score") or 0)
    cluster = normalize_label(e.get("cluster_suggested_label")) if e.get("cluster_suggested_label") else ""
    human = normalize_label(e.get("human_label")) if e.get("human_label") else ""
    ow = normalize_label(e.get("owlv2_crop_top1_label")) if e.get("owlv2_crop_top1_label") else ""
    vf = normalize_label(e.get("verifier_positive_best_label") or e.get("verifier_top1_label")) if (e.get("verifier_positive_best_label") or e.get("verifier_top1_label")) else ""
    dco, cco, oco, vco = fine_to_coarse(dino), fine_to_coarse(cluster) if cluster else "", fine_to_coarse(ow) if ow else "", fine_to_coarse(vf) if vf else ""

    if human:
        return {"final_fine_label": human, "final_coarse_label": fine_to_coarse(human), "display_label": display_label(human), "label_source": "human_label", "fine_uncertain": False, "coarse_conflict": False, "qwen_needed": False, "quality_status": "high_quality", "conflict_type": "none", "decision_reasons": ["human label present"]}
    if e.get("objectness_status") == "fake_object":
        return {"final_fine_label": "background_candidate", "final_coarse_label": "background", "display_label": "background:background_candidate", "label_source": "objectness_gate", "fine_uncertain": False, "coarse_conflict": False, "qwen_needed": False, "quality_status": "rejected", "conflict_type": "fake_object", "decision_reasons": e.get("objectness_reasons", [])}

    final, source, conflict = dino, "dino_initial", "none"
    qwen, quality, fine_uncertain, coarse_conflict = False, "high_quality", False, False
    strong_supports = []
    if _strong_owlv2_support(e, dco):
        strong_supports.append("owlv2")
    if _strong_verifier_support(e, dco):
        strong_supports.append("verifier")

    weak_same_label_agreement = (
        ow == dino and vf == dino and
        (float(e.get("owlv2_crop_top1_score") or 0.0) < MIN_OWLV2_SUPPORT_SCORE or
         float(e.get("verifier_positive_best_score") or e.get("verifier_top1_score") or 0.0) < MIN_VERIFIER_SUPPORT_SCORE)
    )
    noise_cluster = _is_noise_cluster(e.get("cluster_id"))
    high_risk_edge = dino in HIGH_RISK_EDGE_LABELS
    low_dino = dino_score < MIN_DINO_HIGH_CONF

    if strong_supports:
        source = "dino_supported"
        reasons.append(f"DINO coarse strongly supported by {strong_supports}")
    elif dino_score < 0.30 and oco and vco and oco == vco and oco != "background" and (
        float(e.get("owlv2_crop_top1_score") or 0.0) >= MIN_OWLV2_SUPPORT_SCORE or
        float(e.get("verifier_positive_best_score") or e.get("verifier_top1_score") or 0.0) >= MIN_VERIFIER_SUPPORT_SCORE
    ):
        final = ow or vf
        source = "owlv2_verifier_supported"
        reasons.append("low-confidence DINO; OWLv2 and verifier agree with sufficient support")

    if cluster and cco != dco:
        conflict = "cluster_disagrees_with_detector"
    if ow and oco not in {"", "unknown", "background"} and oco != dco:
        conflict = "owlv2_disagrees_with_detector"
    if vf and vco not in {"", "unknown", "background"} and vco != dco:
        conflict = "verifier_disagrees_with_detector"
    coarses = {x for x in [dco, cco, oco, vco] if x and x != "unknown"}
    if len(coarses) >= 3:
        conflict, qwen, quality, coarse_conflict = "severe_coarse_conflict", True, "uncertain", True

    if cluster and cco == dco and cluster != dino:
        conflict, fine_uncertain = "fine_conflict_same_coarse", True

    # New conservative guard: if all models weakly agree on a thin/high-risk label,
    # do not call it high quality. This catches cutting-board/sink/counter edges
    # being labeled as knives or utensils.
    if high_risk_edge and (low_dino or noise_cluster or weak_same_label_agreement) and len(strong_supports) < 2:
        conflict = "edge_fragment_possible_false_positive"
        quality = "uncertain"
        qwen = True
        reasons.append("high-risk thin/edge-like label with weak support; possible board/sink/counter edge false positive")

    # Noise cluster plus low detector confidence should never be high quality unless
    # both crop verifiers strongly support the same coarse class.
    if noise_cluster and low_dino and len(strong_supports) < 2:
        conflict = "noise_cluster_low_confidence"
        quality = "uncertain"
        qwen = True
        reasons.append("proposal comes from noise/outlier cluster with low detector confidence")

    if e.get("verifier_top1_is_negative") and float(e.get("verifier_margin") or 0) > 0.05:
        quality = "rejected" if e.get("objectness_score", 0) < 0.55 else "uncertain"
        qwen = quality == "uncertain"
        conflict = "fake_object" if quality == "rejected" else "qwen_needed"
        reasons.append("verifier top1 is negative with margin")

    if e.get("objectness_status") == "uncertain_object" and quality == "high_quality":
        quality = "uncertain"
    return {"final_fine_label": final, "final_coarse_label": fine_to_coarse(final), "display_label": display_label(final), "label_source": source, "fine_uncertain": fine_uncertain, "coarse_conflict": coarse_conflict, "qwen_needed": qwen, "quality_status": quality, "conflict_type": conflict, "decision_reasons": reasons or ["default DINO/SAM2 proposal retained"]}


## 11. Run verification

In [ ]:
results = []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    rowd = row.to_dict()
    frame, crop, mask, bbox = load_frame(rowd), load_existing_crop(rowd), load_existing_mask(rowd), get_existing_bbox(rowd)
    masked = make_masked_crop(frame, bbox, mask) if USE_MASKED_CROP and frame is not None and bbox is not None and mask is not None else crop
    obj = objectness_gate(rowd)
    ow = run_owlv2_crop_verifier(crop, CANDIDATE_LABELS)
    vf = verify_crop_zero_shot(masked if USE_MASKED_CROP else crop, CANDIDATE_LABELS)
    evidence = {"dino_label": rowd["norm_dino_label"], "dino_score": rowd["norm_dino_score"], "cluster_suggested_label": rowd["norm_cluster_suggested_label"], "human_label": rowd["norm_human_label"], "cluster_id": rowd.get("norm_cluster_id"), **obj, **ow, **vf}
    fusion = choose_final_label(evidence)
    results.append({"proposal_id": rowd["norm_proposal_id"], "frame_path": rowd["norm_frame_path"], "crop_path": rowd["norm_crop_path"], "mask_path": rowd["norm_mask_path"], "bbox_xyxy": rowd["norm_bbox_xyxy"], "cluster_key": rowd["norm_cluster_key"], "cluster_id": rowd["norm_cluster_id"], "dino_label": rowd["norm_dino_label"], "dino_score": rowd["norm_dino_score"], "cluster_suggested_label": rowd["norm_cluster_suggested_label"], **obj, **ow, **vf, **fusion})
results_df = pd.DataFrame(results)
display(results_df[["proposal_id", "dino_label", "objectness_status", "verifier_top1_label", "final_fine_label", "quality_status", "qwen_needed"]].head())

## 12. Qwen / VLM arbitration placeholder

In [ ]:

QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
QWEN_MAX_NEW_TOKENS = 512
FREE_VERIFIER_MODELS_BEFORE_QWEN = True

# Qwen is memory-hungry. The OWLv2/CLIP models are no longer needed after
# section 11, so release them before loading Qwen to avoid slow CPU offload.
if RUN_QWEN and FREE_VERIFIER_MODELS_BEFORE_QWEN:
    import gc
    print("Releasing OWLv2/CLIP verifier models before loading Qwen...")
    try:
        if "owlv2_state" in globals():
            owlv2_state["model"] = None
            owlv2_state["processor"] = None
        if "verifier_state" in globals():
            verifier_state["model"] = None
            verifier_state["processor"] = None
        for _name in ["owlv2_model", "owlv2_processor", "verifier_model", "verifier_processor"]:
            if _name in globals():
                globals()[_name] = None
        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            print("CUDA cache cleared before Qwen load.")
    except Exception as _cleanup_exc:
        print("Qwen pre-load cleanup warning:", _cleanup_exc)

qwen_model = None
qwen_processor = None
qwen_load_error = ""

if RUN_QWEN:
    try:
        from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
        print(f"Loading Qwen model {QWEN_MODEL_ID} on {device}; this can take 1-3 minutes the first time...")
        dtype = torch.float16 if device == "cuda" else torch.float32
        qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            QWEN_MODEL_ID,
            torch_dtype=dtype,
            device_map="auto" if device == "cuda" else None,
            local_files_only=LOCAL_FILES_ONLY,
        ).eval()
        if device == "cpu":
            qwen_model = qwen_model.to(device)
        print("Qwen model weights loaded; loading processor...")
        qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID, local_files_only=LOCAL_FILES_ONLY)
        print("Qwen loaded:", QWEN_MODEL_ID)
    except Exception as exc:
        qwen_load_error = f"{type(exc).__name__}: {exc}"
        print("Qwen unavailable:", qwen_load_error)


def qwen_prompt_from_evidence(evidence: dict) -> str:
    return f"""
You are arbitrating one existing DINO+SAM2 object proposal from an egocentric kitchen/industrial video frame.

Important:
- DINO/SAM2 already produced this bbox/mask/crop. Do not propose new detections unless the crop clearly misses the object.
- First judge whether this proposal is a real object or fake/background.
- Then judge whether model disagreement may be caused by bad mask/crop quality: under-segmentation, over-segmentation, merged instances, background false positive, or unclear crop.
- If mask/crop quality is too bad to identify the object, do NOT force a label. Raise an alert.
- If it is a real object and mask quality is acceptable, arbitrate using:
  1. DINO/SAM2 label and score
  2. OWLv2 crop result
  3. CLIP/SigLIP top-k result
  4. cluster suggested label if available
  5. your own visual judgment
  6. relative context in the full frame. For example, an object at the sink edge may plausibly be a faucet.

Evidence JSON:
{json.dumps(evidence, ensure_ascii=False, default=str, indent=2)}

Return JSON only with this schema:
{{
  "is_object": true,
  "objectness_decision": "real_object | uncertain_object | fake_object",
  "mask_issue": "none | under_segmented | over_segmented | merged_instances | background_false_positive | crop_too_unclear | mask_missing",
  "mask_issue_severity": "none | low | medium | high",
  "needs_mask_alert": false,
  "context_reasoning": "short explanation using full-frame/crop context and location",
  "model_conflict_summary": "short explanation of DINO/OWLv2/CLIP/cluster disagreement",
  "arbitrated_label": null,
  "arbitrated_coarse_label": null,
  "decision": "accept_label | relabel | uncertain | reject | mask_alert",
  "confidence": 0.0,
  "reason": "short final reason"
}}
""".strip()


def parse_qwen_json(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`").removeprefix("json").strip()
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start >= 0 and end > start:
        cleaned = cleaned[start:end+1]
    return json.loads(cleaned)


def run_qwen_arbitration(frame, crop, masked_crop, evidence_dict):
    if not RUN_QWEN or qwen_model is None or qwen_processor is None:
        return None
    images = []
    if frame is not None:
        images.append(frame)
    if crop is not None:
        images.append(crop)
    if masked_crop is not None:
        images.append(masked_crop)
    content = [{"type": "image", "image": im} for im in images]
    content.append({"type": "text", "text": qwen_prompt_from_evidence(evidence_dict)})
    messages = [{"role": "user", "content": content}]
    try:
        text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = qwen_processor(text=[text], images=images if images else None, return_tensors="pt").to(device)
        with torch.inference_mode():
            generated = qwen_model.generate(**inputs, max_new_tokens=QWEN_MAX_NEW_TOKENS, do_sample=False)
        trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated)]
        output = qwen_processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        parsed = parse_qwen_json(output)
        parsed["raw_qwen_output"] = output
        return parsed
    except Exception as exc:
        return {"qwen_error": f"{type(exc).__name__}: {exc}"}


def save_qwen_needed_sample(row_result: dict, qwen_result: dict | None = None):
    pid = row_result["proposal_id"]
    out = QWEN_DIR / f"proposal_{int(pid):07d}"
    out.mkdir(parents=True, exist_ok=True)
    row = sample_df[sample_df["norm_proposal_id"] == pid].iloc[0].to_dict()
    crop, frame, mask, bbox = load_existing_crop(row), load_frame(row), load_existing_mask(row), get_existing_bbox(row)
    masked = make_masked_crop(frame, bbox, mask) if frame is not None and mask is not None and bbox is not None else None
    if frame is not None: frame.save(out / "full_frame.jpg")
    if crop is not None: crop.save(out / "crop.jpg")
    if masked is not None: masked.save(out / "masked_crop.jpg")
    payload = {"evidence": row_result, "qwen_result": qwen_result}
    (out / "evidence.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return str(out)


def apply_qwen_to_result(result: dict, qwen_result: dict | None) -> dict:
    updated = dict(result)
    updated.update({
        "qwen_ran": qwen_result is not None,
        "qwen_objectness_decision": None,
        "qwen_mask_issue": None,
        "qwen_needs_mask_alert": False,
        "qwen_arbitrated_label": None,
        "qwen_decision": None,
        "qwen_confidence": None,
        "qwen_reason": None,
        "final_after_qwen_label": result.get("final_fine_label"),
        "quality_after_qwen": result.get("quality_status"),
    })
    if not qwen_result:
        return updated
    updated["qwen_objectness_decision"] = qwen_result.get("objectness_decision")
    updated["qwen_mask_issue"] = qwen_result.get("mask_issue")
    updated["qwen_needs_mask_alert"] = bool(qwen_result.get("needs_mask_alert"))
    updated["qwen_arbitrated_label"] = qwen_result.get("arbitrated_label")
    updated["qwen_decision"] = qwen_result.get("decision")
    updated["qwen_confidence"] = qwen_result.get("confidence")
    updated["qwen_reason"] = qwen_result.get("reason") or qwen_result.get("qwen_error")
    decision = str(qwen_result.get("decision") or "")
    conf = float(qwen_result.get("confidence") or 0.0)
    if decision == "mask_alert" or qwen_result.get("needs_mask_alert"):
        updated["quality_after_qwen"] = "uncertain"
        updated["qwen_needed"] = True
    elif decision == "reject" or qwen_result.get("objectness_decision") == "fake_object":
        updated["final_after_qwen_label"] = "background_candidate"
        updated["quality_after_qwen"] = "rejected"
        updated["qwen_needed"] = False
    elif decision in {"relabel", "accept_label"} and qwen_result.get("arbitrated_label") and conf >= 0.65:
        label = normalize_label(qwen_result["arbitrated_label"])
        updated["final_after_qwen_label"] = label
        updated["quality_after_qwen"] = "high_quality" if conf >= 0.75 else "uncertain"
        updated["qwen_needed"] = False if conf >= 0.75 else True
    elif decision == "uncertain":
        updated["quality_after_qwen"] = "uncertain"
        updated["qwen_needed"] = True
    return updated

qwen_outputs = []
for result in tqdm(results, desc="Qwen arbitration/export"):
    if result.get("qwen_needed") and result.get("objectness_status") in {"real_object", "uncertain_object"} and result.get("quality_status") != "rejected":
        row = sample_df[sample_df["norm_proposal_id"] == result["proposal_id"]].iloc[0].to_dict()
        frame, crop, mask, bbox = load_frame(row), load_existing_crop(row), load_existing_mask(row), get_existing_bbox(row)
        masked = make_masked_crop(frame, bbox, mask) if frame is not None and bbox is not None and mask is not None else None
        qwen_result = run_qwen_arbitration(frame, crop, masked, result)
        export_path = save_qwen_needed_sample(result, qwen_result)
        updated = apply_qwen_to_result(result, qwen_result)
        updated["qwen_export_path"] = export_path
        qwen_outputs.append(updated)
    else:
        updated = apply_qwen_to_result(result, None)
        updated["qwen_export_path"] = ""
        qwen_outputs.append(updated)

results = qwen_outputs
results_df = pd.DataFrame(results)
print("qwen_needed/exported:", sum(bool(r.get("qwen_export_path")) for r in results))
if RUN_QWEN:
    display(results_df[["proposal_id", "dino_label", "conflict_type", "qwen_decision", "qwen_mask_issue", "qwen_arbitrated_label", "qwen_confidence", "final_after_qwen_label", "quality_after_qwen"]].head(20))


## 13. Visualization

In [ ]:

def _fmt_score(value):
    try:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return "-"
        return f"{float(value):.3f}"
    except Exception:
        return str(value)


def _fmt_top5(labels, scores=None):
    if isinstance(labels, str):
        try:
            labels = ast.literal_eval(labels)
        except Exception:
            labels = [labels]
    if isinstance(scores, str):
        try:
            scores = ast.literal_eval(scores)
        except Exception:
            scores = []
    labels = labels or []
    scores = scores or []
    pairs = []
    for i, label in enumerate(labels[:5]):
        score = _fmt_score(scores[i]) if i < len(scores) else "-"
        pairs.append(f"{label} {score}")
    return ", ".join(pairs) if pairs else "-"


def _debug_lines(result):
    final_after_qwen = result.get("final_after_qwen_label", result.get("final_fine_label"))
    quality_after_qwen = result.get("quality_after_qwen", result.get("quality_status"))
    qwen_line = "Qwen: not run"
    if result.get("qwen_ran"):
        qwen_line = (
            f"Qwen: decision={result.get('qwen_decision')} | object={result.get('qwen_objectness_decision')} | "
            f"mask_issue={result.get('qwen_mask_issue')} | label={result.get('qwen_arbitrated_label')} | "
            f"conf={_fmt_score(result.get('qwen_confidence'))}"
        )
    return [
        f"Proposal: {result.get('proposal_id')} | cluster={result.get('cluster_key') or result.get('cluster_id')}",
        f"DINO/SAM2: label={result.get('dino_label')} | score={_fmt_score(result.get('dino_score'))}",
        f"Cluster: suggested={result.get('cluster_suggested_label') or '-'}",
        f"Objectness: {result.get('objectness_status')} | score={_fmt_score(result.get('objectness_score'))} | mask_quality={_fmt_score(result.get('mask_quality_score'))}",
        f"OWLv2 crop: top1={result.get('owlv2_crop_top1_label')} | score={_fmt_score(result.get('owlv2_crop_top1_score'))}",
        f"OWLv2 top5: {_fmt_top5(result.get('owlv2_crop_top5_labels'), result.get('owlv2_crop_top5_scores'))}",
        f"CLIP/SigLIP: top1={result.get('verifier_top1_label')} | score={_fmt_score(result.get('verifier_top1_score'))} | margin={_fmt_score(result.get('verifier_margin'))}",
        f"CLIP/SigLIP top5: {_fmt_top5(result.get('verifier_top5_labels'), result.get('verifier_top5_scores'))}",
        qwen_line,
        f"Pre-Qwen final: {result.get('display_label')} | quality={result.get('quality_status')} | conflict={result.get('conflict_type')} | qwen_needed={result.get('qwen_needed')}",
        f"FINAL: label={final_after_qwen} | quality={quality_after_qwen}",
        f"Reason: {result.get('qwen_reason') or result.get('decision_reasons')}",
    ]


def draw_debug(result):
    pid = result["proposal_id"]
    row = sample_df[sample_df["norm_proposal_id"] == pid].iloc[0].to_dict()
    frame, crop, mask, bbox = load_frame(row), load_existing_crop(row), load_existing_mask(row), get_existing_bbox(row)
    masked = make_masked_crop(frame, bbox, mask) if frame is not None and bbox is not None and mask is not None else None
    fig, axes = plt.subplots(1, 4, figsize=(22, 5.6), gridspec_kw={"width_ratios": [1.25, 1.0, 1.0, 2.4]})
    for ax in axes:
        ax.axis("off")
    if frame is not None:
        axes[0].imshow(frame); axes[0].set_title("frame + existing bbox")
        if bbox is not None:
            x1, y1, x2, y2 = bbox
            axes[0].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="yellow", linewidth=2))
    if crop is not None:
        axes[1].imshow(crop); axes[1].set_title("existing crop")
    if masked is not None:
        axes[2].imshow(masked); axes[2].set_title("masked crop")
    lines = _debug_lines(result)
    y = 1.0
    for line in lines:
        weight = "bold" if line.startswith("FINAL:") else "normal"
        color = "#0b5" if line.startswith("FINAL:") else "black"
        axes[3].text(0, y, line, va="top", ha="left", fontsize=9.2, fontweight=weight, color=color, wrap=True)
        y -= 0.078 if len(line) < 110 else 0.105
    out = DEBUG_DIR / f"{int(pid):07d}.jpg"
    fig.tight_layout()
    fig.savefig(out, dpi=145)
    plt.close(fig)
    return str(out)

if SAVE_DEBUG_VIS:
    results_df["debug_vis_path"] = [draw_debug(r) for r in tqdm(results_df.to_dict("records"))]
else:
    results_df["debug_vis_path"] = ""
print("debug images:", results_df["debug_vis_path"].astype(bool).sum())


## 14. Export results

In [ ]:
jsonl_path = OUTPUT_ROOT / "cluster_existing_proposal_verification_results.jsonl"
csv_path = OUTPUT_ROOT / "cluster_existing_proposal_verification_results.csv"
with jsonl_path.open("w", encoding="utf-8") as fh:
    for row in results_df.to_dict("records"):
        fh.write(json.dumps(row, ensure_ascii=False, default=str) + "\\n")
results_df.to_csv(csv_path, index=False, encoding="utf-8")
summary = {
    "metadata_path": str(metadata_path_resolved),
    "cluster_summary_path": str(cluster_summary_path_resolved),
    "selected_cluster_key": selected_cluster_key,
    "selected_cluster_id": selected_cluster_id,
    "selected_cluster_size": int(len(cluster_df)),
    "sampled_count": int(len(sample_df)),
    "objectness_counts": results_df["objectness_status"].value_counts().to_dict(),
    "quality_counts": results_df["quality_status"].value_counts().to_dict(),
    "final_coarse_counts": results_df["final_coarse_label"].value_counts().to_dict(),
    "conflict_counts": results_df["conflict_type"].value_counts().to_dict(),
    "qwen_needed": int(results_df["qwen_needed"].sum()),
    "fake_or_rejected": int(((results_df["objectness_status"] == "fake_object") | (results_df["quality_status"] == "rejected")).sum()),
    "low_confidence_dino_preserved": int(((results_df["dino_score"] < 0.30) & (results_df["label_source"].str.startswith("dino"))).sum()),
    "cluster_disagreements": int((results_df["conflict_type"] == "cluster_disagrees_with_detector").sum()),
    "verifier_disagreements": int((results_df["conflict_type"] == "verifier_disagrees_with_detector").sum()),
    "qwen_ran_count": int(results_df.get("qwen_ran", pd.Series(False, index=results_df.index)).fillna(False).sum()),
    "qwen_mask_alert_count": int(results_df.get("qwen_needs_mask_alert", pd.Series(False, index=results_df.index)).fillna(False).sum()),
}
(OUTPUT_ROOT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame([summary]).to_csv(OUTPUT_ROOT / "summary.csv", index=False)
print("wrote", jsonl_path)
print("wrote", csv_path)

## 15. Summary tables

In [ ]:

display(pd.Series(summary["objectness_counts"], name="objectness").to_frame())
display(pd.Series(summary["quality_counts"], name="quality").to_frame())
display(pd.Series(summary["final_coarse_counts"], name="final_coarse").to_frame())
display(pd.Series(summary["conflict_counts"], name="conflict").to_frame())
print("qwen_needed:", summary["qwen_needed"])
print("fake_or_rejected:", summary["fake_or_rejected"])
print("low_confidence_dino_preserved:", summary["low_confidence_dino_preserved"])
print("cluster_disagreements:", summary["cluster_disagreements"])
print("verifier_disagreements:", summary["verifier_disagreements"])
if "qwen_ran" in results_df.columns:
    qwen_ran_df = results_df[results_df["qwen_ran"].fillna(False).astype(bool)]
    print("\nQwen arbitration summary")
    print("qwen_ran:", len(qwen_ran_df))
    if len(qwen_ran_df):
        display(qwen_ran_df["qwen_decision"].value_counts(dropna=False).rename("qwen_decision_count").to_frame())
        display(qwen_ran_df["qwen_mask_issue"].value_counts(dropna=False).rename("qwen_mask_issue_count").to_frame())
        rescued = qwen_ran_df[
            (qwen_ran_df.get("quality_status", "") == "uncertain")
            & (qwen_ran_df.get("quality_after_qwen", "") == "high_quality")
        ]
        relabeled = qwen_ran_df[
            qwen_ran_df.get("final_after_qwen_label", qwen_ran_df.get("final_fine_label")).astype(str).str.lower()
            != qwen_ran_df.get("final_fine_label").astype(str).str.lower()
        ]
        alerts = qwen_ran_df[qwen_ran_df.get("qwen_needs_mask_alert", False).fillna(False).astype(bool)]
        print("rescued uncertain -> high_quality:", len(rescued))
        print("qwen relabeled:", len(relabeled))
        print("qwen mask alerts:", len(alerts))
        display(qwen_ran_df[[c for c in [
            "proposal_id", "dino_label", "conflict_type", "qwen_decision", "qwen_objectness_decision",
            "qwen_mask_issue", "qwen_needs_mask_alert", "qwen_arbitrated_label", "qwen_confidence",
            "final_fine_label", "final_after_qwen_label", "quality_status", "quality_after_qwen", "qwen_reason"
        ] if c in qwen_ran_df.columns]])


review_cols = [
    "proposal_id", "dino_label", "dino_score", "cluster_suggested_label",
    "objectness_status", "owlv2_crop_top1_label", "owlv2_crop_top1_score",
    "verifier_top1_label", "verifier_top1_score", "verifier_margin",
    "final_fine_label", "display_label", "quality_status", "conflict_type",
    "qwen_needed", "debug_vis_path",
]
display(results_df[[c for c in review_cols if c in results_df.columns]])

from html import escape
contact_html = OUTPUT_ROOT / "debug_contact_sheet.html"
card_html = []
for rec in results_df.to_dict("records"):
    img = rec.get("debug_vis_path") or ""
    rel = ""
    if img and Path(img).exists():
        try:
            rel = Path(img).resolve().relative_to(OUTPUT_ROOT.resolve()).as_posix()
        except Exception:
            rel = Path(img).resolve().as_uri()
    title = f"proposal {rec.get('proposal_id')} | {rec.get('display_label')} | {rec.get('quality_status')} | {rec.get('conflict_type')}"
    info = {
        "dino": rec.get("dino_label"),
        "dino_score": rec.get("dino_score"),
        "cluster": rec.get("cluster_suggested_label"),
        "objectness": rec.get("objectness_status"),
        "owlv2": rec.get("owlv2_crop_top1_label"),
        "verifier": rec.get("verifier_top1_label"),
        "final": rec.get("display_label"),
        "qwen_needed": rec.get("qwen_needed"),
        "reasons": rec.get("decision_reasons"),
    }
    card_html.append(
        '<div class="card">'
        + '<div class="title">' + escape(title) + '</div>'
        + ('<a href="' + escape(rel) + '" target="_blank"><img src="' + escape(rel) + '"></a>' if rel else '<div>No debug image</div>')
        + '<pre>' + escape(json.dumps(info, ensure_ascii=False, default=str, indent=2)) + '</pre>'
        + '</div>'
    )
html = (
    '<!doctype html><html><head><meta charset="utf-8"><title>Cluster proposal verification contact sheet</title>'
    '<style>body{font-family:Segoe UI,Arial,sans-serif;background:#111;color:#eee;margin:20px}'
    '.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(520px,1fr));gap:16px}'
    '.card{border:1px solid #444;padding:10px;background:#1b1b1b}.title{font-weight:600;margin-bottom:8px}'
    'img{width:100%;height:auto;border:1px solid #333}pre{white-space:pre-wrap;font-size:12px;color:#ccc}</style>'
    '</head><body><h1>Cluster Existing Proposal Verification Contact Sheet</h1>'
    f'<p>Total sampled proposals: {len(results_df)}</p><div class="grid">' + ''.join(card_html) + '</div></body></html>'
)
contact_html.write_text(html, encoding="utf-8")
print("Wrote contact sheet:", contact_html)

PAGE_SIZE = 10
paths = [Path(p) for p in results_df["debug_vis_path"].dropna() if p and Path(p).exists()]
print(f"Debug images available: {len(paths)} / {len(results_df)}")
for start in range(0, len(paths), PAGE_SIZE):
    subset = paths[start:start + PAGE_SIZE]
    fig, axes = plt.subplots(len(subset), 1, figsize=(16, 5 * len(subset)))
    if len(subset) == 1:
        axes = [axes]
    for ax, path in zip(axes, subset):
        ax.imshow(Image.open(path)); ax.axis("off"); ax.set_title(path.name)
    plt.tight_layout()
    plt.show()


## 16. Interpretation notes

- Existing DINO+SAM2 proposal is the starting point.
- This notebook does not rerun DINO/SAM2.
- This notebook does not recluster.
- Objectness Gate prevents fake proposals from forcing OWLv2/CLIP/Qwen to choose a wrong label.
- OWLv2 and CLIP/SigLIP are crop-level verifiers/suggesters.
- Cluster evidence is auxiliary and must not overwrite detector labels by itself.
- Qwen is only for hard real-object conflicts.
- Only `high_quality` samples should be considered for precision-first training export.